# Chapter 5: Recursion and the Call Stack

The programs from Chapter 5 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## A Function That Calls Itself

`factorial.py`

In [1]:
"""Recursion: a function that calls itself on a smaller problem."""


def factorial(n):
    if n <= 1:  # base case: small enough to answer directly
        return 1
    return n * factorial(n - 1)  # recursive case: a smaller problem


def total(items):
    """Sum a list: the first item plus the sum of the rest."""
    if not items:
        return 0
    return items[0] + total(items[1:])


print(factorial(5))
print(factorial(20))
print(total([3, 1, 4, 1, 5, 9]))

120
2432902008176640000
23


## The Call Stack

`call_stack.py`

In [2]:
"""Show each call and return, indented by the depth of the stack."""


def factorial(n, depth=0):
    pad = "  " * depth
    print(f"{pad}factorial({n}) called")
    result = 1 if n <= 1 else n * factorial(n - 1, depth + 1)
    print(f"{pad}factorial({n}) returns {result}")
    return result


factorial(4)

factorial(4) called
  factorial(3) called
    factorial(2) called
      factorial(1) called
      factorial(1) returns 1
    factorial(2) returns 2
  factorial(3) returns 6
factorial(4) returns 24


24

## The Recursion Limit

`recursion_limit.py`

In [3]:
"""Python stops recursion that goes too deep."""

import sys


def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)


print("limit:", sys.getrecursionlimit())
print(depth(900))
try:
    depth(100_000)
except RecursionError:
    print("RecursionError: too deep")

limit: 1000
900
RecursionError: too deep


## Fibonacci and the Explosion of Calls

`fibonacci.py`

In [4]:
"""Fibonacci three ways, counting the calls each one makes."""

from functools import cache

calls = 0


def fib_naive(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


@cache  # remembers every answer it has computed
def fib_memo(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)


def fib_loop(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


for n in [10, 20, 25]:
    calls = 0
    answer = fib_naive(n)
    naive_calls = calls
    calls = 0
    fib_memo.cache_clear()
    assert fib_memo(n) == answer == fib_loop(n)
    print(
        f"fib({n}) = {answer:,}: naive {naive_calls:,} calls, "
        f"memoized {calls} calls"
    )

fib(10) = 55: naive 177 calls, memoized 11 calls
fib(20) = 6,765: naive 21,891 calls, memoized 21 calls
fib(25) = 75,025: naive 242,785 calls, memoized 26 calls


`bench_fibonacci.py`

In [5]:
"""Time the naive and the memoized Fibonacci."""

from functools import cache

import bench


def fib_naive(n):
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)


def fib_memo(n):
    @cache
    def f(k):
        return k if k < 2 else f(k - 1) + f(k - 2)

    return f(n)  # a fresh cache every call, so each call does the work


sizes = [16, 18, 20, 22, 24]
results = {
    "naive": bench.doubling(fib_naive, lambda n: (n,), sizes),
    "memoized": bench.doubling(fib_memo, lambda n: (n,), sizes),
}
bench.table(results)

         n           naive        memoized
        16         78.7 us         3.83 us
        18     207 us x2.6    4.12 us x1.1
        20     539 us x2.6    4.43 us x1.1
        22    1.43 ms x2.7    4.98 us x1.1
        24     3.7 ms x2.6    4.89 us x1.0


## Recursion on Nested Data

`flatten.py`

In [6]:
"""Recursion fits nested data: flatten lists inside lists."""


def flatten(items):
    out = []
    for x in items:
        if isinstance(x, list):
            out.extend(flatten(x))  # a smaller problem of the same kind
        else:
            out.append(x)
    return out


def flatten_loop(items):
    """The same with an explicit stack instead of recursion."""
    out, stack = [], [iter(items)]
    while stack:
        for x in stack[-1]:
            if isinstance(x, list):
                stack.append(iter(x))
                break
            out.append(x)
        else:  # this list is finished
            stack.pop()
    return out


nested = [1, [2, [3, 4], 5], [], [[6]], 7]
print(flatten(nested))
print(flatten_loop(nested))

deep = [0]
for i in range(1, 5000):
    deep = [deep, i]  # nested 5,000 levels deep
print(len(flatten_loop(deep)))  # the loop has no depth limit

[1, 2, 3, 4, 5, 6, 7]
[1, 2, 3, 4, 5, 6, 7]
5000
